# Viabilitat del Torn de Matí (Nivell 1)
Aquest quadern respon a la pregunta: **Hi cap el matí?**

L'objectiu és calcular el pic màxim de sessions simultànies que demanen anar al matí (d'acord al model on 1r i 2n hi van) i comparar-ho amb la disponibilitat de l'aulari.

In [1]:
import pandas as pd

# 1. Carregar les dades
sessions = pd.read_csv('data/sessions.csv')
aules = pd.read_csv('data/aules.csv')

# Mostrar breument quantes aules ordinàries tenim
aules_ordinaries = aules[aules['ordinaria'] == 1]
print(f"Total d'aules ordinàries a l'inventari: {len(aules_ordinaries)}")
print("Aules disponibles per franja (descomptant 2 de reserva per la restricció 8): 19")


Total d'aules ordinàries a l'inventari: 21
Aules disponibles per franja (descomptant 2 de reserva per la restricció 8): 19


In [2]:
# 2. Filtrar les sessions pertinents
# - Només torn_requerit = 'mati'
# - Que necessitin espai (sense_espai != 1)
# - Excloem els laboratoris (PLAB) perquè no es fan en aules ordinàries sinó en laboratoris departamentals
mati_sessions = sessions[
    (sessions['torn_requerit'] == 'mati') &
    (sessions['sense_espai'] != 1) &
    (sessions['tipus'] != 'PLAB')
].copy()

print(f"Total de sessions programades al matí que necessiten aula ordinària: {len(mati_sessions)}")


Total de sessions programades al matí que necessiten aula ordinària: 7710


In [3]:
# 3. Funció per calcular el pic màxim per dia (Sweep-line algorithm)
def get_max_simultaneous_by_date(df):
    max_simult_global = 0
    peak_date_global = None
    
    # Agrupar per data exacta al calendari
    for data in df['data'].dropna().unique():
        df_dia = df[df['data'] == data]
        events = []
        
        # Creem esdeveniments d'inici (+1) i final (-1) de classe
        for _, row in df_dia.iterrows():
            if pd.notna(row['h_ini']) and pd.notna(row['h_fi']):
                events.append((row['h_ini'], 1))  # Ocupa una aula
                events.append((row['h_fi'], -1))  # Allibera una aula
                
        # Ordenar cronològicament.
        # Si l'hora coincideix, el -1 va abans que l'1 per no sobrecomptar si una classe entra quan l'altra surt.
        events.sort(key=lambda x: (x[0], x[1]))
        
        current_simult = 0
        max_dia = 0
        for time, change in events:
            current_simult += change
            if current_simult > max_dia:
                max_dia = current_simult
                
        # Actualitzem el rècord global del semestre
        if max_dia > max_simult_global:
            max_simult_global = max_dia
            peak_date_global = data
            
    return max_simult_global, peak_date_global


In [4]:
# 4. Executar el càlcul per a cada semestre i concloure
c1_mati = mati_sessions[mati_sessions['semestre'] == 'C/1']
c2_mati = mati_sessions[mati_sessions['semestre'] == 'C/2']

pic_c1, data_c1 = get_max_simultaneous_by_date(c1_mati)
pic_c2, data_c2 = get_max_simultaneous_by_date(c2_mati)

print("=== RESULTATS DE VIABILITAT DEL MATÍ ===")
print(f"1r Semestre: {pic_c1} classes simultànies màximes (assolit el {data_c1})")
print(f"2n Semestre: {pic_c2} classes simultànies màximes (assolit el {data_c2})")
print("------------------------------------------")
if max(pic_c1, pic_c2) <= 19:
    print("✅ CONCLUSIÓ: El matí HI CAP. El pic de cap dels dos semestres supera les 19 aules ordinàries disponibles.")
else:
    print("❌ CONCLUSIÓ: El matí NO HI CAP. Se superen les 19 aules ordinàries disponibles en algun punt.")


=== RESULTATS DE VIABILITAT DEL MATÍ ===
1r Semestre: 16 classes simultànies màximes (assolit el 2026-12-14)
2n Semestre: 15 classes simultànies màximes (assolit el 2027-03-16)
------------------------------------------
✅ CONCLUSIÓ: El matí HI CAP. El pic de cap dels dos semestres supera les 19 aules ordinàries disponibles.
